# ПЗ 2 — Нарезка видео на кадры

Скачиваем видео по ссылке, нарезаем на кадры с заданным шагом, сохраняем в Google Drive.

In [ ]:
!pip install opencv-python-headless yt-dlp tqdm -q

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

BASE_DRIVE  = '/content/drive/MyDrive/cv-frames'
VIDEO_DIR   = f'{BASE_DRIVE}/видио'
FRAMES_DIR  = f'{BASE_DRIVE}/кадры'
RESULTS_DIR = f'{BASE_DRIVE}/результаты'

for d in [VIDEO_DIR, FRAMES_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"папки готовы")

In [ ]:
# вставьте ссылку на видео (YouTube, VK, Rutube и др.)
VIDEO_URL = 'https://www.youtube.com/watch?v=ВСТАВЬТЕ_ССЫЛКУ'

video_path = f'{VIDEO_DIR}/video.mp4'

!yt-dlp -f "bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best" \
    -o "{video_path}" \
    --no-playlist \
    "{VIDEO_URL}"

if os.path.exists(video_path):
    size_mb = os.path.getsize(video_path) / 1024 / 1024
    print(f'скачано: {size_mb:.1f} MB — {video_path}')
else:
    print('не скачалось, проверьте ссылку')

In [ ]:
# альтернатива — загрузить файл вручную
# from google.colab import files
# import shutil
# uploaded = files.upload()
# name = list(uploaded.keys())[0]
# video_path = f'{VIDEO_DIR}/{name}'
# shutil.move(name, video_path)

In [ ]:
import cv2

cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print('видео не открылось, проверьте файл')
else:
    fps   = cap.get(cv2.CAP_PROP_FPS)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f'fps: {fps:.1f}, кадров: {total}, длина: {total/fps:.1f} сек')

In [ ]:
from tqdm.notebook import tqdm

FRAME_STEP = 30

frame_idx = 0
saved = 0

for _ in tqdm(range(total), desc='нарезка'):
    ret, frame = cap.read()
    if not ret:
        break
    if frame_idx % FRAME_STEP == 0:
        cv2.imwrite(f'{FRAMES_DIR}/frame_{frame_idx:06d}.jpg', frame)
        saved += 1
    frame_idx += 1

cap.release()
print(f'сохранено {saved} кадров в {FRAMES_DIR}')

In [ ]:
from IPython.display import Image, display

frames = sorted(f for f in os.listdir(FRAMES_DIR) if f.endswith('.jpg'))[:3]
for f in frames:
    display(Image(filename=f'{FRAMES_DIR}/{f}', width=400))